# DFL 인플레이 H/A TranSPORTmer

Colab에 이 노트북을 업로드하고 **런타임 → 런타임 유형 변경 → GPU → 모두 실행**을 누르세요. Drive 연결을 승인한 뒤 셀 3에서 **DFL_7matches_Parquet.zip**을 한 번 선택하세요. ZIP은 저장소의 `prepare_colab_cache.py`로 공개 DFL Parquet에서 만들 수 있습니다. Drive에 저장한 캐시는 다음 실행부터 재사용합니다.

배치 24 벤치마크 뒤 개발 6경기 OOF, 오답 수집, 공통 워밍업, 홀드아웃 B/C 비교가 자동으로 진행됩니다. 분할 seed는 2027이고 홀드아웃은 J03WOH입니다.

결과와 체크포인트는 **내 드라이브 → DFL_TranSPORTmer → seed2027_local_cache**에 저장됩니다. 세션이 끊기면 같은 노트북에서 **모두 실행**을 누르세요. Drive 저장 없이 실행하려면 셀 1에서 `USE_DRIVE = False`를 쓰면 됩니다.

DFL 자료는 Bassek et al. (2025)의 공개 데이터입니다. 노트북에는 MTAH 볼 결측률과 길이별 집계가 포함됩니다. 보간 비율 25%는 시뮬레이션 가정입니다. [DFL 원본 및 CC BY 4.0](https://springernature.figshare.com/articles/dataset/An_integrated_dataset_of_spatiotemporal_and_event_data_in_elite_soccer/28196177)

In [ ]:
# @title 1. GPU 및 저장 위치 준비
import json
import shutil
import subprocess
import sys
from pathlib import Path
from time import perf_counter

import torch

USE_DRIVE = True
if not torch.cuda.is_available():
    raise RuntimeError("런타임 → 런타임 유형 변경에서 GPU를 선택하고 모두 실행하세요.")

started = perf_counter()
ROOT = Path("/content/dfl_transportmer")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = Path("/content/drive/MyDrive/DFL_TranSPORTmer")
else:
    SAVE = Path("/content/DFL_TranSPORTmer")
OUTPUT = SAVE / "seed2027_local_cache"
CACHE = SAVE / "parquet_local"
for folder in (ROOT, OUTPUT, CACHE):
    folder.mkdir(parents=True, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)} | PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f"출력: {OUTPUT}")

In [ ]:
# @title 2. 포함된 실험 코드 풀기
# 원본 모듈 목록은 아래 압축을 푼 뒤 출력됩니다.
import base64
import io
import zipfile

bundle = '''
...
'''
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle))) as archive:
    archive.extractall(ROOT)
    print("\n".join(archive.namelist()))
profile = OUTPUT / "ball_profile.json"
if not profile.exists():
    shutil.copyfile(ROOT / profile.name, profile)
elif json.loads(profile.read_text()) != json.loads((ROOT / profile.name).read_text()):
    raise ValueError(f"다른 볼 결측 집계가 저장되어 있습니다. 실행 폴더를 확인하세요: {OUTPUT}")

In [ ]:
# @title 3. Mac에서 만든 공개 DFL 캐시 불러오기
import hashlib

import pyarrow.parquet as pq

archive_name = "DFL_7matches_Parquet.zip"
matches = ("J03WMX", "J03WN1", "J03WOH", "J03WOY", "J03WPY", "J03WQQ", "J03WR9")
members = {f"{match}_{kind}.parquet" for match in matches for kind in ("tracking", "events", "meta")}

def cache_valid():
    try:
        return all(pq.read_metadata(CACHE / name).num_rows > 0 for name in members)
    except (OSError, ValueError):
        return False

cache_marker = CACHE / "archive_sha256.txt"
saved_archive_sha256 = cache_marker.read_text().strip() if cache_marker.is_file() else None
archive_sha256 = saved_archive_sha256
if not (OUTPUT / "holdout_comparison.csv").is_file():
    data_started = perf_counter()
    if not (saved_archive_sha256 and cache_valid()):
        from google.colab import files

        local_archive = Path("/content") / archive_name
        if not local_archive.is_file():
            print(f"파일 선택에서 {archive_name}을 업로드하세요.", flush=True)
            uploaded = files.upload()
            payload = uploaded.get(archive_name)
            if payload is None:
                raise ValueError(f"{archive_name}을 선택하고 셀 3부터 다시 실행하세요.")
            stream = io.BytesIO(payload)
        else:
            stream = local_archive.open("rb")

        with stream:
            archive_sha256 = hashlib.file_digest(stream, "sha256").hexdigest()
            if saved_archive_sha256 and saved_archive_sha256 != archive_sha256:
                raise ValueError("이전 실행에 사용한 ZIP을 선택하세요. 다른 ZIP은 새 저장 폴더에서 실행하세요.")
            stream.seek(0)
            with zipfile.ZipFile(stream) as packed:
                if set(packed.namelist()) != members or len(packed.namelist()) != len(members):
                    raise ValueError("ZIP 안에 DFL 7경기 Parquet 21개가 있어야 합니다.")
                for name in sorted(members):
                    temporary = CACHE / (name + ".tmp")
                    with packed.open(name) as source, temporary.open("wb") as target:
                        shutil.copyfileobj(source, target)
                    temporary.replace(CACHE / name)
        if not cache_valid():
            raise ValueError("Parquet 파일을 읽을 수 없습니다.")
        cache_marker.write_text(archive_sha256 + "\n")
        print(f"로컬 DFL 7경기 Parquet 저장 완료: {CACHE}", flush=True)
    else:
        print(f"Drive에 저장된 DFL 7경기 Parquet 재사용: {CACHE}", flush=True)
    print(f"DFL 준비 {perf_counter() - data_started:.1f}초", flush=True)

In [ ]:
# @title 4. 배치 24 벤치마크 및 학습
import shlex
from datetime import datetime, timezone

report = OUTPUT / "holdout_comparison.csv"
command = [sys.executable, "-u", "src/run_transportmer_ha.py", "--device", "cuda",
           "--seed", "2027", "--cache", str(CACHE), "--output", str(OUTPUT)]
reproduction = ["cd " + shlex.quote(str(ROOT)),
                shlex.join(command + ["--stage", "benchmark"]),
                shlex.join(command + ["--stage", "run"])]
if not report.is_file():
    (OUTPUT / "reproduce.sh").write_text("set -eu\n" + "\n".join(reproduction) + "\n")
    log_path = OUTPUT / "colab_run.log"
    error_path = OUTPUT / "colab_error.json"
    about = {
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "dfl_local_archive_sha256": archive_sha256,
        "setup_and_data_seconds_this_session": perf_counter() - started,
    }

    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        print(json.dumps(about), file=log)
        for stage in ("benchmark", "run"):
            invocation = command + ["--stage", stage]
            print(shlex.join(invocation), file=log)
            with subprocess.Popen(invocation, cwd=ROOT, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                status = process.wait()
            if status:
                error_path.write_text(json.dumps({
                    "stage": stage, "command": shlex.join(invocation), "exit_code": status,
                    "log": str(log_path), "resume": "Run all cells again after addressing the error.",
                }, indent=2) + "\n")
                raise RuntimeError(f"{stage} 실패 · 세부 내용: {log_path}")
    error_path.unlink(missing_ok=True)
else:
    print(f"완료된 결과를 찾았습니다: {report}")

In [ ]:
# @title 5. OOF와 홀드아웃 결과 보기
import pandas as pd
from IPython.display import display

print("6경기 OOF 평활화 평균")
display(pd.read_csv(OUTPUT / "oof_smoothing_mean.csv"))
print("DFL 홀드아웃 1경기 B/C 비교")
comparison = pd.read_csv(OUTPUT / "holdout_comparison.csv")
if set(comparison["condition"]) != {"B", "C"} or len(comparison) != 2:
    raise ValueError("홀드아웃 결과에서 B/C 두 행을 찾지 못했습니다.")
display(comparison)
print(f"결과 폴더: {OUTPUT}")
print("분할 manifest.json | 오답 error_ledger.csv | 시간 timing.json | 재현 reproduce.sh")